# GOA model: TEMPO vs p-PQME (TCL2) vs TCL234
One notebook, five steps: **(1)** set the GOA parameters, **(2)** print the Hamiltonian and bath, **(3)** build the $(\alpha,\omega_h)$ frame grid, **(4)** run TEMPO and the TCL2/TCL234 frame scan, **(5)** compare (per-frame errors, oracle / typical / reference-free median, plots).

Units: $\hbar=k_B=1$, $\omega_c=1$. Initial state $|D\rangle\langle D|\otimes\rho_B$ (bath thermal on the donor surface). Frame filter $W_h(\omega)=1-\exp[-(\omega/\omega_h)^\alpha]$.
Run from the `notebooks/` folder (uses the package `goa_pqme/` one level up; the equations are derived step by step in `PPQME_Code_Walkthrough.ipynb`); needs `numpy scipy pandas matplotlib`, and `oqupy` for TEMPO (install it in a separate environment if it changes your numpy; otherwise set `LOAD_TEMPO_CSV` below).

In [ ]:
import sys, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('..'))            # package root (goa_pqme/)
import goa_pqme as gp
from goa_pqme.scan import run_grid, print_model
from goa_pqme.metrics import analyse
from goa_pqme.tempo import run_tempo
%matplotlib inline

## 1. Parameters (edit this cell)

In [ ]:
# --- GOA model ---------------------------------------------------------
Gamma    = 0.2        # electronic coupling
omega_DA = 0.0        # donor energy (acceptor sits at E_r); try 0.28125 for omega_DA = E_r at y0 = 0.75
beta     = np.inf     # inverse temperature in units of 1/omega_c (np.inf = zero temperature)
y0       = 0.75       # displacement of the primary mode, E_r = 2*Omega^2*y0^2
Omega, N, omega_c, eta = 0.5, 50, 1.0, 1.0   # primary-mode frequency, number of modes, cutoff, secondary-mode strength

# --- time window and numerics -------------------------------------------
TMAX      = 24.0      # final time (Gamma=1 cases: use 6)
NODE_DT   = 3.0       # spacing of the sparse nodes where K3, K4 are computed (interpolated in between)
NQ, PANEL = 4, 6.0    # Gauss-Legendre points per panel and panel width of the K3/K4 quadrature
TEMPO_DT, TEMPO_EPS = 0.2, 1e-6      # TEMPO Trotter step and truncation threshold: CHECK CONVERGENCE for your case

# --- frame grid -----------------------------------------------------------
FULL_GRID = False     # True: 10 alpha x 12 omega_h = 120 frames (paper grid, ~11 s per frame per core); False: small test grid
NPROC     = 2         # parallel processes for the frame scan
LOAD_TEMPO_CSV = None # e.g. '../data/trajectories/G0p2_OD0_Binf_y0p75.csv' to skip running TEMPO (columns t_au/t, dP)
RESULTS = 'results'   # output folder (frames cached here, re-running skips finished frames)
os.makedirs(RESULTS, exist_ok=True)

p = gp.ModelParams(Gamma=Gamma, omega_DA=omega_DA, beta=beta, y0=y0, Omega=Omega, N=N, omega_c=omega_c, eta=eta)
NUM = gp.Numerics(node_dt=NODE_DT, nq=NQ, panel_h=PANEL, dt=0.05, tmax=TMAX)

## 2. Hamiltonian and bath
Lab-frame Hamiltonian, reorganisation energy, and (for one example frame) the transformed system Hamiltonian $\tilde H_S$ with $B=\langle V_f\rangle$ and $\lambda_p$.

In [ ]:
EX_ALPHA, EX_WH = 2.0, 0.3      # example frame to print
om, R, Er = print_model(p, EX_ALPHA, EX_WH)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].stem(om, R, basefmt=' '); ax[0].set_xlabel(r'$\omega_q$'); ax[0].set_ylabel(r'$R_q$'); ax[0].set_title('mode displacements')
ax[1].stem(om, 0.5*om**2*R**2, basefmt=' '); ax[1].set_xlabel(r'$\omega_q$'); ax[1].set_title(r'reorganisation energy per mode $\frac{1}{2}\omega_q^2R_q^2$')
for a_, w_ in [(0.5, .05), (2, .3), (4, 1.), (10, 2.)]:
    ax[2].plot(np.sort(om), gp.polaron_filter(np.sort(om), a_, w_), label=r'$\alpha$=%g, $\omega_h$=%g' % (a_, w_))
ax[2].set_xlabel(r'$\omega$'); ax[2].set_title(r'filter $f=W_h(\omega)$ (fraction polaron-transformed)'); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 3. Frame grid

In [ ]:
if FULL_GRID:
    ALPHAS = np.array([0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.5, 8.0, 10.5])               # edit to match your own grid (paper: 10 alphas in 0.5-10.5)
    OMEGAHS = np.array([0.01, 0.02, 0.05, 0.095, 0.15, 0.29, 0.5, 0.8, 1.2, 2.0, 3.0, 4.0])  # 12 omega_h in 0.01-4
else:
    ALPHAS  = np.array([0.5, 2.0, 4.0, 8.0])
    OMEGAHS = np.array([0.05, 0.3, 1.0, 3.0])
print(len(ALPHAS)*len(OMEGAHS), 'frames;  approx. cost %.0f min on %d process(es)' % (len(ALPHAS)*len(OMEGAHS)*11/60/NPROC, NPROC))

## 4a. TEMPO reference
Same discrete bath, same initial state. If `oqupy` is not installed, set `LOAD_TEMPO_CSV` in the parameter cell.

In [ ]:
tempo_file = os.path.join(RESULTS, gp.tag(p) + '_tempo_T%g_dt%g_eps%g.csv' % (TMAX, TEMPO_DT, TEMPO_EPS))
if LOAD_TEMPO_CSV:
    ref = pd.read_csv(LOAD_TEMPO_CSV).rename(columns={'t_au': 't'}); ref = ref[ref.t <= TMAX + 1e-9].reset_index(drop=True)
elif os.path.exists(tempo_file):
    ref = pd.read_csv(tempo_file)
else:
    ref = run_tempo(p, tmax=TMAX, dt=TEMPO_DT, epsrel=TEMPO_EPS); ref.to_csv(tempo_file, index=False)
plt.figure(figsize=(5, 3)); plt.plot(ref.t, ref.dP, 'k'); plt.xlabel('t'); plt.ylabel(r'$\Delta P=P_D-P_A$'); plt.title('TEMPO'); plt.show()

## 4b. TCL2 and TCL234 on every frame
For each frame: $\mathcal K_2$ (dense), $\mathcal K_3,\mathcal K_4$ at the sparse nodes, then propagation of TCL2, TCL2+K3, TCL2+K4 and TCL234. Finished frames are cached in `results/frames_cache/`.

In [ ]:
res = run_grid(p, ALPHAS, OMEGAHS, NUM, nproc=NPROC, cache=os.path.join(RESULTS, 'frames_cache'))

## 5. Comparison
Error of a frame $=\frac12\,\mathrm{RMSE}(\Delta P-\Delta P_{\rm TEMPO})$. A frame is *stable* if $\Delta P$ is finite with $|\Delta P|\le1.6$.
**Oracle** = best frame against TEMPO (needs the reference); **typical** = median frame error; **RF** = median over stable frames of $\Delta P(t)$ (reference-free).

In [ ]:
df, summ, tt, curves = analyse(res, ref)
df.to_csv(os.path.join(RESULTS, gp.tag(p) + '_frame_errors.csv'), index=False)
table = pd.DataFrame({m: {'stable frames': '%d/%d' % (s['n_stable'], s['n_frames']), 'oracle err': s['oracle'], 'oracle frame (alpha,omega_h)': s['oracle_frame'],
                          'typical frame err': s['typical'], 'reference-free (RF) err': s['RF']} for m, s in summ.items()}).T
display(table)
df.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for a_, m in zip(ax, ('TCL2', 'TCL234')):
    for (a, w, name), y in curves.items():
        if name == m and df[(df.alpha == a) & (df.omega_h == w)]['stable_' + m].iloc[0]: a_.plot(tt, y, color='0.8', lw=0.7)
    a_.plot(tt, summ[m]['median_curve'], 'C0' if m == 'TCL2' else 'C1', lw=2, label='%s RF median (err %.4f)' % (m, summ[m]['RF']))
    if summ[m]['oracle_frame'] is not None:
        a_.plot(tt, curves[(*summ[m]['oracle_frame'], m)], 'g--', lw=1.2, label='oracle frame (%.3g, %.3g): %.4f' % (*summ[m]['oracle_frame'], summ[m]['oracle']))
    a_.plot(ref.t[:len(tt)], ref.dP[:len(tt)], 'k', lw=2, label='TEMPO'); a_.set_title(m + ' (grey = individual stable frames)'); a_.set_xlabel('t'); a_.set_ylabel(r'$\Delta P$'); a_.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, gp.tag(p) + '_curves.png'), dpi=150); plt.show()

In [ ]:
# error maps over the frame grid
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for a_, m in zip(ax[:2], ('TCL2', 'TCL234')):
    Z = np.full((len(ALPHAS), len(OMEGAHS)), np.nan)
    for r in df.itertuples():
        if getattr(r, 'stable_' + m): Z[list(ALPHAS).index(r.alpha), list(OMEGAHS).index(r.omega_h)] = getattr(r, 'err_' + m)
    im = a_.imshow(np.log10(Z), origin='lower', aspect='auto', cmap='viridis'); a_.set_title('log10 error, ' + m + ' (white = unstable)')
    a_.set_xticks(range(len(OMEGAHS))); a_.set_xticklabels(OMEGAHS); a_.set_yticks(range(len(ALPHAS))); a_.set_yticklabels(ALPHAS); a_.set_xlabel(r'$\omega_h$'); a_.set_ylabel(r'$\alpha$'); plt.colorbar(im, ax=a_)
Z = np.full((len(ALPHAS), len(OMEGAHS)), np.nan)
for r in df.itertuples(): Z[list(ALPHAS).index(r.alpha), list(OMEGAHS).index(r.omega_h)] = r.r4
im = ax[2].imshow(Z, origin='lower', aspect='auto', cmap='magma'); ax[2].set_title(r'$r_4=K_4/K_2$ (rms)'); plt.colorbar(im, ax=ax[2])
ax[2].set_xticks(range(len(OMEGAHS))); ax[2].set_xticklabels(OMEGAHS); ax[2].set_yticks(range(len(ALPHAS))); ax[2].set_yticklabels(ALPHAS); ax[2].set_xlabel(r'$\omega_h$')
plt.tight_layout(); plt.show()
print('Does TCL234 beat TCL2?  RF: %.4f -> %.4f ;  typical: %.4f -> %.4f ;  oracle: %.4f -> %.4f' % (summ['TCL2']['RF'], summ['TCL234']['RF'], summ['TCL2']['typical'], summ['TCL234']['typical'], summ['TCL2']['oracle'], summ['TCL234']['oracle']))

### Notes
* Populations only. The lab-frame coherence readout ($B\tilde\rho_{DA}+Q_1$) is available as `gp.run_frame(p, alpha, omega_h, NUM, coherence=True).coherence` (`goa_pqme.dynamics.q1`) but is not part of this notebook.
* The inhomogeneous TCL terms are omitted: they vanish for $|D\rangle\langle D|\otimes\rho_B$ with the acceptor-only transformation used here.
* K3/K4 are evaluated at nodes every `NODE_DT` and spline-interpolated; for strong coupling (`Gamma` near 1) use `TMAX=6` and a smaller `NODE_DT` (e.g. 1) as the dynamics is fast.
* Reproduces the shipped scan: e.g. `Gamma=0.2, omega_DA=0, beta=inf, y0=0.75`, frame (0.5, 0.05): TCL2 error 0.0779, TCL234 error 6.13 (diverged), r4 = 1.39.
* Check TEMPO convergence (`TEMPO_DT`, `TEMPO_EPS`) for every new parameter set.